# BG-forecasting — transformer arm on Colab

Runs the transformer cell under **the same protocol as the RNN grid**, to answer
R1 W2 and R3 #5: the claim that the findings "apply unchanged to newer
architectures" is currently asserted rather than tested. One architecture
replicated under the identical protocol is what R3 asked for.

This reuses `run_seed_major.sh` — the same script the RNN grid uses — with
`MODELS="transformer"`, so the derived config, the preflight, the resume logic
and the output layout are identical. Nothing here is transformer-specific except
the config it derives from.

---

### The one thing to decide first

`configs/full_transformer_30min.yaml` sets `include_feature_engineering: true`,
which appends `hour_sin`/`hour_cos` and gives **F=6**. The RNN grid runs at
**F=4**. Section 2 makes that an explicit choice:

- `FEATURE_SET = "F4"` — patch the config to match the RNN grid. This is what
  "the same protocol" means, and the only setting that supports a like-for-like
  comparison in the paper.
- `FEATURE_SET = "F6"` — keep the config as written, if the extra time features
  are deliberate for attention. The result then answers "does a transformer do
  better", not "does the finding hold across architectures".

`run_seed_major.sh` preflights this flag and refuses to run at F=6, so `"F6"`
also disables that guard. Decide before you spend the GPU hour, not after.

## 1 · Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- edit these to match your Drive ---------------------------------------
DRIVE_DATA    = "/content/drive/MyDrive/ohiot1dm"
DRIVE_RESULTS = "/content/drive/MyDrive/bg-results"
REPO_DIR      = "/content/BG-forecasting"

SOURCE     = "git"
REPO_URL   = "https://github.com/beatriz-fulgencio/BG-forecasting.git"
BRANCH     = "bench2"
DRIVE_REPO = "/content/drive/MyDrive/BG-forecasting"

MODEL    = "transformer"
HORIZONS = [30]              # 30 min matches the cell the paper discusses
SEEDS    = [41, 42, 43]

# "F4" matches the RNN grid; "F6" keeps the config's own time features.
FEATURE_SET = "F4"
# ---------------------------------------------------------------------------

import os, pathlib
for d in (DRIVE_RESULTS, f"{DRIVE_RESULTS}/experiments", f"{DRIVE_RESULTS}/logs"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print(f"Drive ready. {MODEL} at {HORIZONS} min, seeds {SEEDS}, {FEATURE_SET}.")

## 2 · Get the code, and settle the feature set

In [ ]:
import shutil, subprocess, sys, pathlib, yaml

if SOURCE == "git":
    if not pathlib.Path(REPO_DIR).is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("WARNING: git pull --ff-only failed; this checkout may be stale.\n"
                  + (pull.stderr or pull.stdout).strip() + "\n")
elif SOURCE == "drive":
    if not pathlib.Path(REPO_DIR).is_dir():
        shutil.copytree(DRIVE_REPO, REPO_DIR)
else:
    raise ValueError("SOURCE must be 'git' or 'drive'")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
os.chmod("run_seed_major.sh", 0o755)

CONFIGS = [f"configs/full_{MODEL}_{h}min.yaml" for h in HORIZONS]
missing = [f for f in ["run_seed_major.sh"] + CONFIGS if not pathlib.Path(f).is_file()]
if missing:
    raise SystemExit("Missing from this checkout:\n  " + "\n  ".join(missing))

# Apply the feature-set decision to the working copy. This edits the checkout,
# not the repository: nothing is committed, and a re-clone starts over.
for cfg_path in CONFIGS:
    p = pathlib.Path(cfg_path)
    text = p.read_text()
    pre = yaml.safe_load(text)["preprocessing"]
    want_fe = (FEATURE_SET == "F6")
    if pre["unimodal"] is not False:
        raise SystemExit(f"{cfg_path}: unimodal is not false -> glucose only (F=1)")
    if pre["include_feature_engineering"] != want_fe:
        p.write_text(text.replace(
            f"include_feature_engineering: {str(pre['include_feature_engineering']).lower()}",
            f"include_feature_engineering: {str(want_fe).lower()}", 1))
        print(f"  {cfg_path}: include_feature_engineering -> {want_fe}")
    arch = yaml.safe_load(p.read_text())["model"]["architecture"]
    print(f"  {cfg_path}: {arch}")

# run_seed_major.sh preflights include_feature_engineering and refuses F=6.
# At FEATURE_SET = "F6" that guard is deliberately being overridden, so relax it
# in the working copy -- and say so, because it is the check that stops an
# accidental F=6 run of the RNN grid.
if FEATURE_SET == "F6":
    script = pathlib.Path("run_seed_major.sh")
    text = script.read_text()
    guarded = "'^  include_feature_engineering: *false'"
    if guarded in text:
        script.write_text(text.replace(guarded, "'^  include_feature_engineering: *(false|true)'", 1))
        print("  run_seed_major.sh: F=6 allowed for this run (preflight relaxed in the checkout)")

head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %cd %s", "--date=short"],
                      capture_output=True, text=True).stdout.strip()
print(f"\nRepo ready at {REPO_DIR}" + (f"\n  commit: {head}" if head else ""))

import torch
if torch.cuda.is_available():
    print(f"  device: cuda -> {torch.cuda.get_device_name(0)}")
else:
    print("  WARNING: no CUDA device. The configs set device: cuda, so every cell will fail.")

## 3 · Stage the OhioT1DM data

Identical to the training notebook: 24 files, copied individually so a partial
stage is repaired rather than skipped.

In [ ]:
import shutil, pathlib

COHORT = {"2018": [559, 563, 570, 575, 588, 591],
          "2020": [540, 544, 552, 567, 584, 596]}

dst = pathlib.Path(REPO_DIR) / "data" / "raw" / "ohiot1dm"
src = pathlib.Path(DRIVE_DATA)
if not src.is_dir():
    raise SystemExit(f"{DRIVE_DATA} not found.")

copied = 0
for release, patients in COHORT.items():
    for mode, suffix in (("train", "training"), ("test", "testing")):
        (dst / release / mode).mkdir(parents=True, exist_ok=True)
        for pid in patients:
            name = f"{pid}-ws-{suffix}.xml"
            target, source = dst / release / mode / name, src / release / mode / name
            if target.is_file() or not source.is_file():
                continue
            shutil.copy2(source, target)
            copied += 1

missing = [f"{r}/{m}/{p}-ws-{s}.xml"
           for r, pats in COHORT.items()
           for m, s in (("train", "training"), ("test", "testing"))
           for p in pats if not (dst / r / m / f"{p}-ws-{s}.xml").is_file()]
print(f"{copied} file(s) copied; {len(sorted(dst.glob('*/*/*.xml')))}/24 staged.")
if missing:
    raise SystemExit("Missing OhioT1DM file(s):\n  " + "\n  ".join(missing))

## 4 · Why the earlier transformer runs stalled

`results/transformer_tuning/` holds the trial records from the tuning sweep. If
the arm stalled for a reason that is recorded there — diverged loss, OOM, a
config the runner rejected — it is cheaper to read it than to rediscover it at
hour one. Skip this cell if there is nothing to diagnose.

In [ ]:
import json, pathlib

tuning = pathlib.Path(REPO_DIR) / "results" / "transformer_tuning"
trials = sorted(tuning.glob("trial_*.json")) if tuning.is_dir() else []
print(f"{len(trials)} tuning trial(s) in {tuning}\n")

rows = []
for t in trials:
    try:
        d = json.loads(t.read_text())
    except Exception as exc:
        print(f"  {t.name}: unreadable ({exc})")
        continue
    rows.append((t.name, d))

for name, d in rows[:10]:
    keys = [k for k in ("status", "error", "val_loss", "best_val_loss", "mae", "params") if k in d]
    print(f"  {name}: " + ", ".join(f"{k}={d[k]}" for k in keys) or f"  {name}: {list(d)[:6]}")

# Any previous transformer run left in the results tree?
exp = pathlib.Path(REPO_DIR) / "results" / "experiments"
prior = [p for p in exp.glob("*/resolved_config.yaml") if "transformer" in p.read_text()] if exp.is_dir() else []
print(f"\n{len(prior)} prior transformer run(s) in results/experiments")
for p in prior:
    done = (p.parent / "aggregate_metrics.json").is_file()
    print(f"  {p.parent.name}: {'complete' if done else 'INCOMPLETE'}")

## 5 · Restore earlier progress, then check the plan

Prints what would run without running anything. Finished cells are skipped, so
this notebook is safe to re-run after a disconnect.

In [ ]:
import pathlib, shutil, subprocess, sys, os

EXP_DIR = pathlib.Path(REPO_DIR) / "results" / "experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)

restored = 0
for d in sorted(pathlib.Path(f"{DRIVE_RESULTS}/experiments").glob("experiment_*")):
    target = EXP_DIR / d.name
    if not target.exists():
        shutil.copytree(d, target)
        restored += 1
print(f"Restored {restored} experiment dir(s) from Drive.\n")

env = dict(os.environ, DRY_RUN="1",
           SEEDS=" ".join(map(str, SEEDS)),
           MODELS=MODEL,
           HORIZONS=" ".join(map(str, HORIZONS)),
           PYTHON=sys.executable)
print(subprocess.run(["./run_seed_major.sh"], cwd=REPO_DIR, env=env,
                     capture_output=True, text=True).stdout)

## 6 · Run

Seed-major, one cell at a time, mirroring to Drive after each. A transformer
cell is heavier per step than a GRU cell, so budget more than the ~20 min a GRU
cell takes — measure the first one before assuming the rest.

In [ ]:
import subprocess, shutil, pathlib, time, sys, os, collections

def mirror_to_drive():
    dst = pathlib.Path(f"{DRIVE_RESULTS}/experiments")
    copied = 0
    for d in EXP_DIR.glob("experiment_*"):
        if not (d / "aggregate_metrics.json").is_file():
            continue
        if (dst / d.name).exists():
            continue
        shutil.copytree(d, dst / d.name)
        copied += 1
    return copied

KEEP = ("[", "seed ", "  ok ", "  FAILED", "Preflight", "Nothing", "Done:", "config:", "ERROR", "Traceback")

def run_cell(seed, horizon):
    name = f"full_{MODEL}_{horizon}min_seed{seed}"
    env = dict(os.environ, SEEDS=str(seed), MODELS=MODEL, HORIZONS=str(horizon),
               PYTHON=sys.executable, LOG_DIR=f"{DRIVE_RESULTS}/logs")
    t0 = time.time()
    proc = subprocess.Popen(["./run_seed_major.sh"], cwd=REPO_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    tail = collections.deque(maxlen=40)
    for line in proc.stdout:
        tail.append(line.rstrip())
        if any(k in line for k in KEEP):
            print(line.rstrip(), flush=True)
    rc = proc.wait()
    if rc != 0:
        print(f"    --- last {len(tail)} line(s) of {name} ---", flush=True)
        for line in tail:
            print("    " + line, flush=True)
        print(f"    --- full log: {DRIVE_RESULTS}/logs/{name}.log ---", flush=True)
    copied = mirror_to_drive()
    print(f"    {'OK ' if rc == 0 else 'FAIL'} {name} in {(time.time()-t0)/60:.0f} min"
          f"  (mirrored {copied} dir(s))", flush=True)
    return rc

failures = []
for seed in SEEDS:
    print(f"\n{'='*62}\n SEED {seed}\n{'='*62}", flush=True)
    for horizon in HORIZONS:
        if run_cell(seed, horizon) != 0:
            failures.append(f"full_{MODEL}_{horizon}min_seed{seed}")

print("\nFailed cells:", failures if failures else "none")

## 7 · Merge the seeds

Same merge the RNN grid uses, so the transformer cell lands in the same shape
the analysis notebook expects: one parent per cell, three seeds, cross-seed
means and Student-t intervals.

In [ ]:
import subprocess, sys, shutil, pathlib

args = [sys.executable, "merge_seed_runs.py",
        "--seeds", *map(str, SEEDS),
        "--models", MODEL,
        "--horizons", *map(str, HORIZONS),
        "--copy"]

print(subprocess.run(args + ["--dry-run"], cwd=REPO_DIR, capture_output=True, text=True).stdout)
result = subprocess.run(args + ["--force"], cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

merged_src = pathlib.Path(REPO_DIR) / "results" / "experiments_merged"
if merged_src.is_dir():
    shutil.copytree(merged_src, pathlib.Path(f"{DRIVE_RESULTS}/experiments_merged"),
                    dirs_exist_ok=True)
    print(f"Merged tree copied to {DRIVE_RESULTS}/experiments_merged")

## 8 · Status

When every seed reads `done`, add this cell to the analysis notebook's `MODELS`
list and re-run it — the transformer then flows through the same stages, tests
and corrections as the RNN cells, which is the comparison R3 #5 asked for.

In [ ]:
import pathlib, yaml

done = set()
for resolved in EXP_DIR.glob("*/resolved_config.yaml"):
    if (resolved.parent / "aggregate_metrics.json").is_file():
        try:
            done.add(yaml.safe_load(resolved.read_text())["experiment"]["name"])
        except Exception:
            pass

print(f"{'cell':<26}" + "".join(f"seed {s:<6}" for s in SEEDS))
print("-" * (26 + 11 * len(SEEDS)))
for horizon in HORIZONS:
    cell = f"full_{MODEL}_{horizon}min"
    marks = "".join(f"{'done' if f'{cell}_seed{s}' in done else '--':<11}" for s in SEEDS)
    print(f"{cell:<26}{marks}")

merged = pathlib.Path(REPO_DIR) / "results" / "experiments_merged"
print(f"\nmerged cells: {sorted(p.name for p in merged.glob(f'full_{MODEL}_*')) if merged.is_dir() else []}")
print(f"feature set used: {FEATURE_SET}")